# Hand Pipeline — Colab

```
WEBCAM -> OpenCV frame
   -> 1. DETECT    where is the hand?       box + score
   -> 2. SEGMENT   which pixels are hand?   mask
   -> 3. POSE      21 landmarks             fingertip coords
   -> 4. TRACK     frame to frame           id, trajectory, velocity
   -> gesture / movement
```

## Read this first — Colab is not your laptop

Colab runs on a **remote virtual machine**. That machine has no camera and no screen, so:

| Works locally | On Colab |
|---|---|
| `cv2.VideoCapture(0)` | **Fails** — the VM has no webcam. Use the JavaScript bridge in section 4. |
| `cv2.imshow(...)` | **Fails** — no display. Use `cv2_imshow` instead. |
| `python run_pipeline.py` | **Fails** — it opens a window. Call the stages directly, as below. |

Everything else — the four stages themselves — runs unchanged.


## 1. What files you need

| File | Size | Where it comes from | Needed for |
|---|---|---|---|
| `hand_pipeline_colab.zip` | 46 KB | **You upload it** (section 2) | everything |
| `models/hand_landmarker.task` | 7.5 MB | Cell downloads it automatically | everything |
| `checkpoints/hand_detector.pth` | 8.7 MB | **You upload it**, optional | only the `ssdlite` detector |
| `metadata.mat` | 19 MB | Only for EgoHands work | training / evaluation |
| `_LABELLED_SAMPLES/` | 1.2 GB | Cell downloads it (section 6) | training only |

**For the live demo you need only the first two rows.** MediaPipe needs no checkpoint at all.

Upload targets go in the Colab session's working directory (`/content`), which is wiped when
the runtime disconnects — re-run sections 2 and 3 after a reconnect.


## 2. Install dependencies and upload the code


In [ ]:
!pip install -q mediapipe==0.10.21 'numpy<2'
print('installed — if Colab asks you to restart the runtime, do it, then re-run from here')


In [ ]:
# Upload hand_pipeline_colab.zip (produced in your project root)
from google.colab import files
import zipfile, os

uploaded = files.upload()          # pick hand_pipeline_colab.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall('.')
print('extracted:')
for root, _dirs, fs in os.walk('pipeline'):
    for f in sorted(fs):
        if f.endswith('.py'):
            print('  ', os.path.join(root, f))


## 3. Fetch the MediaPipe model and verify the stages import


In [ ]:
!mkdir -p models
!wget -q -O models/hand_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!ls -lh models/hand_landmarker.task


In [ ]:
import cv2, numpy as np, mediapipe as mp
from pipeline.mediapipe_hands import MediaPipeHands
from pipeline.detector import HandDetector
from pipeline.pose import PoseEstimator
from pipeline.segmenter import HandSegmenter
from pipeline.tracker import HandTracker
from pipeline.gestures import classify
from pipeline.visualize import render

print('opencv   ', cv2.__version__)
print('mediapipe', mp.__version__)
print('all four stages imported OK')


In [ ]:
# Build the pipeline once and reuse it in every later cell.
# Stages 1 and 3 SHARE one MediaPipe runner: HandLandmarker returns the box and the
# 21 landmarks from a single inference, so running it twice would halve the frame
# rate for nothing. Measured locally: 86.3 ms -> 42.9 ms per frame.

import time

shared    = MediaPipeHands('models/hand_landmarker.task', max_hands=4,
                           confidence=0.4, video_mode=False)
detector  = HandDetector(backend='mediapipe', shared=shared)
pose_est  = PoseEstimator(shared=shared)
segmenter = HandSegmenter(method='hybrid')
tracker   = HandTracker()

def run_pipeline_on(frame_bgr, fps=0.0):
    """All four stages on one frame. Returns (annotated_frame, tracks)."""
    now = time.perf_counter()
    detections = detector.detect(frame_bgr)                       # 1 DETECT
    poses      = pose_est.estimate(frame_bgr, detections)         # 3 POSE
    masks      = [segmenter.segment(frame_bgr, d, p)              # 2 SEGMENT
                  for d, p in zip(detections, poses)]
    tracks     = tracker.update(detections, poses, masks, timestamp=now)   # 4 TRACK
    for t in tracks:
        if t.pose is not None:
            t.gesture, t.gesture_confidence = classify(t.pose, t.detection)
    return render(frame_bgr.copy(), tracks, fps), tracks

print('pipeline ready')


## 4. Webcam — the JavaScript bridge

Colab cannot reach your camera from Python. This asks **your browser** for a frame via
`getUserMedia`, base64-encodes it, and hands it to Python. Your browser will ask permission.


In [ ]:
from IPython.display import display, Javascript, Image as IPyImage
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(quality=0.9):
    """Grab one frame from the browser's camera and return it as a BGR array."""
    display(Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Capture';
        div.appendChild(capture);
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => capture.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }
    '''))
    data = eval_js(f'takePhoto({quality})')
    raw = b64decode(data.split(',')[1])
    return cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)

print('take_photo() ready — run the next cell, allow the camera, hold up a hand, press Capture')


In [ ]:
from google.colab.patches import cv2_imshow

frame = take_photo()
frame = cv2.flip(frame, 1)                 # mirror, so it matches what you saw
annotated, tracks = run_pipeline_on(frame)

print(f'{len(tracks)} hand(s) found')
for t in tracks:
    tip = t.pose.points[8] if t.pose is not None else None
    print(f'  #{t.track_id} {t.detection.handedness}  box={t.detection.box}  '
          f'gesture={t.gesture} ({t.gesture_confidence:.0%})  '
          f'index_tip={None if tip is None else tip.round(1).tolist()}')

cv2_imshow(annotated)


## 5. Video — where movement actually becomes measurable

Stage 4 needs **sequences**. A still image has no trajectory and no velocity.

Note this if you plan to use EgoHands for motion: its 100 "labelled frames" per video are
sampled about **0.7 seconds apart** (median gap 21 frames at 30 fps), so consecutive
annotations are not consecutive in time and frame-to-frame tracking on them is meaningless.
Record your own clip, or use any video file.


In [ ]:
# Upload any short .mp4 (a phone clip of your hand moving works well)
from google.colab import files
uploaded = files.upload()
video_path = next(iter(uploaded))
print('uploaded', video_path)


In [ ]:
import time

tracker = HandTracker()      # fresh tracker so ids start clean for this clip
cap = cv2.VideoCapture(video_path)
fps_in = cap.get(cv2.CAP_PROP_FPS) or 30.0
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter('annotated_raw.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps_in, (w, h))

n = 0; started = time.perf_counter(); trajectories = {}
while True:
    ok, frame = cap.read()
    if not ok:
        break
    out, tracks = run_pipeline_on(frame, fps_in)
    for t in tracks:
        trajectories.setdefault(t.track_id, []).append(t.detection.center)
    writer.write(out); n += 1
cap.release(); writer.release()

elapsed = time.perf_counter() - started
print(f'{n} frames in {elapsed:.1f}s -> {n/max(elapsed,1e-6):.1f} fps')
print(f'tracks seen: {sorted(trajectories)}')
for tid, pts in trajectories.items():
    if len(pts) > 1:
        dist = sum(float(np.hypot(b[0]-a[0], b[1]-a[1])) for a, b in zip(pts, pts[1:]))
        print(f'  track #{tid}: {len(pts)} frames, path length {dist:.0f} px')


In [ ]:
# OpenCV's mp4v stream will not play in the browser. Re-encode to H.264 so it does.
!ffmpeg -y -loglevel error -i annotated_raw.mp4 -vcodec libx264 -pix_fmt yuv420p annotated.mp4

from IPython.display import HTML
from base64 import b64encode
data = b64encode(open('annotated.mp4','rb').read()).decode()
HTML(f'<video width=720 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')


## 6. Optional — the EgoHands dataset and training

Only needed if you want to train the SSDlite detector rather than use MediaPipe.

**An honest warning.** A detector trained only on EgoHands boxes faces, elbows and feet on a
webcam. EgoHands is head-mounted footage where the only skin-coloured objects are hands, so the
model learns "skin-coloured blob = hand" — 100% correct on EgoHands, wrong everywhere else.
For webcam use, stay with the MediaPipe detector above.

Indiana University removed the original download; the Internet Archive has the genuine file.


In [ ]:
# ~1.3 GB. Skip unless you are training.
!wget -q --show-progress -O egohands_data.zip \
  'https://web.archive.org/web/20200713164330id_/http://vision.soic.indiana.edu/egohands_files/egohands_data.zip'
!unzip -q -n egohands_data.zip -d .
!ls _LABELLED_SAMPLES | head -5
!echo "video folders: $(ls _LABELLED_SAMPLES | wc -l)  (expect 48)"
!echo "jpeg frames:   $(find _LABELLED_SAMPLES -name '*.jpg' | wc -l)  (expect 4800)"


In [ ]:
# Training also needs metadata.mat and the training scripts from the project root.
# Upload metadata.mat, detection_dataset.py, train_detector.py, get_meta_by.py,
# get_frame_path.py and get_bounding_boxes.py, then:
#
#   !pip install -q torch torchvision scipy pandas tqdm
#   !python train_detector.py --epochs 10
#
# On a Colab T4 this is considerably faster than a laptop. Set
# Runtime -> Change runtime type -> T4 GPU first.
print('see the comments above')


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `NotFoundError` on the .task file | model not downloaded | re-run section 3 |
| Camera cell hangs | browser permission denied | allow camera, reload the page |
| `numpy` version error after install | mediapipe pins `numpy<2` | Runtime → Restart, re-run from section 2 |
| Video shows a black player | mp4v not browser-playable | run the ffmpeg re-encode cell |
| Everything vanishes | runtime disconnected, `/content` wiped | re-run sections 2 and 3 |
| No hands detected | hand too small or out of frame | move closer, improve lighting |
